In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict

# Define time period
start = (datetime.utcnow() - timedelta(days=2)).strftime("%Y-%m-%dT%H:%M:%SZ")

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 9199 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,dnsActive,whoisActive,sha1,url,address,text,md5,sha256,size,indicator
0,6755399446027683,2025-05-07T12:51:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-08T14:24:14Z,3.0,80,INC9045771,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.237.131.164
1,5629499537015529,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-08T14:24:14Z,5.0,99,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.225.189.132
2,5265118,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-08T14:24:14Z,3.0,69,CISA conducted a hunt on IoC's obtained from o...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.153.151.137
3,5265117,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-08T14:24:14Z,3.0,78,CISA conducted a hunt on IoC's obtained from o...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,194.26.192.77
4,4890776,2024-09-13T20:48:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-08T14:24:14Z,3.0,44,Sharing malicious IPs flagged by virustotal an...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.200.13.201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,4500255,2024-01-12T16:42:59Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T07:36:37Z,4.0,30,CISA has observed infrastructure linked to Vol...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,154.91.255.35
9996,4494752,2023-12-27T15:08:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T07:36:37Z,3.0,11,IB-23-20094 New Callisto Group Infrastructure ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.12.89.20
9997,4486719,2023-12-11T16:53:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T07:36:37Z,4.0,7,Researches at multiple threat intel companies ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.51.134.27
9998,4480744,2023-11-27T18:46:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-06T07:36:37Z,4.0,52,***Indicator Sharing - DICOM Scanning***\n---\...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142.4.218.114


In [ ]:
import pandas as pd
from datetime import timedelta

# File paths
observed_data_path = r"C:\\Users\\jaskew\\Documents\\project_repository\\data\\processed\\ProcessedObservedData.csv"

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Load Observed Data
observed_data_df = pd.read_csv(observed_data_path)

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=48)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Apply additional filters
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
recent_tags = recent_tags[recent_tags['observations'] < 15000]

# Filter by dateAdded in the last 30 days
recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'description', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')

# Remove rows where 'all_tags' contains 'I&W'
recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]

recent_tags


,id,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,all_tags,observed_date,partner_count,partners
0,471298,2025-05-08T14:22:52Z,104.237.131.164,4409,Address,2025-05-07 12:51:47+00:00,2025-05-08T14:24:14Z,2025-05-08 00:00:00+00:00,"[DHA Splunk API, VA OIS CSOC CTS Splunk, CMS S...",2025-05-07,4,"CMS, DHA, HRSA, IHS"
8,471298,2025-05-08T14:22:52Z,193.105.73.21,173,Address,2025-05-07 12:51:47+00:00,2025-05-08T14:24:12Z,2025-05-08 00:00:00+00:00,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-07,2,"DHA, OS"
10,471298,2025-05-08T14:22:52Z,37.19.200.131,740,Address,2025-04-23 15:01:06+00:00,2025-05-08T14:24:12Z,2025-05-08 00:00:00+00:00,"[DHA Splunk API, UNC5537, VA OIS CSOC CTS Splu...",2025-05-07,4,"CMS, DHA, HRSA, IHS"
18,30479,2025-05-08T14:23:20Z,45.135.57.192,4696,Address,2025-05-07 12:51:44+00:00,2025-05-08T14:24:07Z,2025-05-08 00:00:00+00:00,"[VA OIS CSOC CTS Splunk, CMS Splunk API, VA CS...",2025-05-07,3,"CMS, HRSA, IHS"
21,35760,2025-05-08T14:21:56Z,193.3.19.2,29,Address,2025-05-07 12:51:46+00:00,2025-05-08T14:24:07Z,2025-05-08 00:00:00+00:00,"[OS Splunk API, FDA Splunk API, Observed, mali...",2025-05-07,2,"FDA, OS"
35,471298,2025-05-08T14:22:52Z,45.33.120.158,3334,Address,2025-05-07 12:51:47+00:00,2025-05-08T14:24:03Z,2025-05-08 00:00:00+00:00,"[DHA Splunk API, VA OIS CSOC CTS Splunk, CMS S...",2025-05-08,4,"CMS, DHA, HRSA, IHS"
39,35760,2025-05-08T14:21:56Z,119.96.59.16,9280,Address,2025-05-07 12:51:46+00:00,2025-05-08T14:05:12Z,2025-05-08 00:00:00+00:00,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-05-08,3,"CMS, HRSA, OS"


In [38]:
import os
import time
import urllib3
import pandas as pd
import requests
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ensure NLTK stopwords are downloaded
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))

# === CONFIG ===
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# === Helper Functions ===

def fetch_virustotal_data(ip):
    """Fetch data from VirusTotal API."""
    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
    headers = {"x-apikey": VT_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error querying VirusTotal API for {ip}: {e}")
        return None

def fetch_otx_data(ip):
    """Fetch data from OTX API."""
    url = f"https://otx.alienvault.io/api/v1/indicators/IPv4/{ip}/general"
    headers = {"X-OTX-API-KEY": OTX_API_KEY}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying OTX API for {ip}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    # Extract the nested dictionary and create new columns
    base_df = pd.json_normalize(df['base_indicator'])

    # Rename columns to avoid conflicts
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    # Drop the original 'base_indicator' column and join the new columns
    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)

    return df

# === Main Script ===

def main(recent_tags):
    """ Main function to fetch data and generate DataFrames. """
    # Ensure 'summary' column exists
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing in the provided DataFrame.")
        return pd.DataFrame(), pd.DataFrame()

    # Extract search terms from 'summary' column
    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Found {len(search_terms)} unique search terms in 'summary' column.")

    # Initialize DataFrames
    vt_data = []
    otx_data = []

    for indicator in search_terms:
        print(f"\nFetching data for: {indicator}")
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        # Extract 'partners' and 'partner_count'
        partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
        partner_count_values = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
        partner_count = partner_count_values[0] if len(partner_count_values) > 0 else "N/A"
        observed_by = partners[0] if len(partners) > 0 else "N/A"

        # === VIRUSTOTAL ===
        vt_data_response = fetch_virustotal_data(indicator)

        if vt_data_response:
            attributes = vt_data_response.get("data", {}).get("attributes", {})
            services = attributes.get("services", [])
            ports = [s.get("port") for s in services if "port" in s]

            vt_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'link': f"https://www.virustotal.com/gui/ip-address/{indicator}",
                'asn': attributes.get('asn'),
                'as_owner': attributes.get('as_owner'),
                'country': attributes.get('country'),
                'network': attributes.get('network'),
                'last_analysis_stats': attributes.get('last_analysis_stats'),
                'reputation': attributes.get('reputation'),
                'total_votes': attributes.get('total_votes'),
                'open_ports': ports,
                'observed_by': observed_by,
                'partner_count': partner_count
            }

            vt_data.append(vt_entry)

        # === OTX ===
        otx_data_response = fetch_otx_data(indicator)

        if otx_data_response:
            otx_entry = {
                'search_term': indicator,
                'timestamp': timestamp,
                'link': f"https://otx.alienvault.com/indicator/ip/{indicator}",
                'base_indicator': otx_data_response.get('base_indicator', {}),
                'reputation': otx_data_response.get('reputation'),
                'geo': otx_data_response.get('geo', {}),
                'whois': otx_data_response.get('whois', {}),
                'open_ports': otx_data_response.get('open_ports', []),
                'observed_by': observed_by,
                'partner_count': partner_count
            }

            otx_data.append(otx_entry)

    # Convert lists to DataFrames
    vt_df = pd.DataFrame(vt_data)
    otx_df = pd.DataFrame(otx_data)

    # Unnest 'base_indicator' in OTX DataFrame
    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(recent_tags)
    print("Script completed successfully.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jaskew\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Found 7 unique search terms in 'summary' column.

Fetching data for: 104.237.131.164

Fetching data for: 193.105.73.21

Fetching data for: 37.19.200.131

Fetching data for: 45.135.57.192

Fetching data for: 193.3.19.2

Fetching data for: 45.33.120.158

Fetching data for: 119.96.59.16
Script completed successfully.


In [41]:
import os
import pandas as pd
from datetime import datetime
from docx import Document

# File paths
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Collect links from VirusTotal
    vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    vt_links.columns = ['search_term', 'vt_links']

    # Collect links from OTX
    otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = "IPv4 Address" if '.' in str(row.get('search_term', '')) else "Domain"
        new_row[2].text = extract_date(row.get('timestamp', 'N/A'))
        new_row[3].text = str(row.get('observed_by', 'N/A'))
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return

    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace `{{sources}}` placeholder outside the table
        for para in doc.paragraphs:
            if "{{sources}}" in para.text:
                all_sources = ', '.join(df['sources'].dropna().unique())
                para.text = para.text.replace("{{sources}}", all_sources)

        # Save the modified document
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report: {e}")

def main(vt_df, otx_df):
    """ Main function to handle report generation. """
    # Combine the DataFrames and drop duplicates
    combined_df = pd.concat([vt_df, otx_df], ignore_index=True).drop_duplicates(subset=['search_term'])

    # Consolidate sources
    sources_df = consolidate_sources(vt_df, otx_df)

    # Merge sources into the combined DataFrame
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Define output file path
    current_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    output_file = os.path.join(OUTPUT_DIR, f"I&W_Report_{current_date}.docx")

    # Fill the template table with data
    fill_word_template(TEMPLATE_PATH, output_file, combined_df)

if __name__ == "__main__":
    main(vt_df, otx_df)



Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\I&W_Report_2025-05-08.docx
